# Phase 4 — Price Rigidity & Promotion Dynamics

## TL;DR
This notebook presents the deterministic Phase 4 artifacts from `src/price_rigidity.py`. Its scope is descriptive price rigidity and promotion dynamics; econometric results are documented separately in `05_econometrics.ipynb`. The unit is store × UPC × week, and every comparison is between the same store–UPC in consecutive calendar weeks with valid prices.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'data/processed/rigidity').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'data/processed/rigidity'
FIG = ROOT / 'report/figures'
manifest = json.loads((OUT / 'phase4_manifest.json').read_text())
assert manifest['phase4_closed'] and all(manifest['validation'].values())
manifest['phase4_gate'], manifest['counts']

## Price-change classification
`UNCHANGED`, `INCREASE`, and `DECREASE` are mutually exclusive and exhaustive. Effective prices are compared as exact reduced rational values of the supplied decimal `PRICE` divided by integer `QTY`; no economic threshold is used.

In [ ]:
changes = pd.read_csv(OUT / 'price_change_summary.csv')
changes[['sample','eligible_comparison_count','unchanged_count','increase_count','decrease_count','price_change_frequency','mean_absolute_log_change_conditional_on_change','median_absolute_log_change_conditional_on_change']]

In [ ]:
display(Image(filename=str(FIG / '10_price_change_frequency.png'), width=760))
display(Image(filename=str(FIG / '13_increase_vs_decrease_magnitudes.png'), width=760))

## Observed price spells and censoring
A gap, missing valid-price observation, price change, or store–UPC boundary ends a run or spell. The first and last spell of every uninterrupted run are censored; only internally bounded spells are marked completed. Observed durations are descriptive, not unbiased structural durations.

In [ ]:
spells = pd.read_csv(OUT / 'price_spell_summary.csv')
spell_summary = spells.loc[spells.record_type.eq('summary')]
display(spell_summary)
display(Image(filename=str(FIG / '11_price_spell_duration.png'), width=820))

## Promotion transitions, entries/exits, and reversals
Blank `SALE` remains `UNFLAGGED`, not confirmed non-promotion. B/C/S remain `DOCUMENTED_PROMOTION`; G/L remain `UNKNOWN_PROMOTION_CODE`. All nine transitions are retained. Entry/exit and MOVE patterns are descriptive associations only.

In [ ]:
transitions = pd.read_csv(OUT / 'promotion_transition_summary.csv')
entry_exit = pd.read_csv(OUT / 'promotion_entry_exit_summary.csv')
movement_share = pd.read_csv(OUT / 'promotion_price_movement_share.csv')
reversals = pd.read_csv(OUT / 'price_reversal_summary.csv')
display(transitions)
display(entry_exit)
display(movement_share)
display(reversals)

In [ ]:
display(Image(filename=str(FIG / '12_promotion_transition_price_changes.png'), width=900))

## Heterogeneity
Entity rows retain sample counts. All 486 CORE UPCs are listed; 17 with zero eligible consecutive comparisons have undefined frequency estimates rather than being silently dropped. Store coverage includes all 93 stores.

In [ ]:
upc = pd.read_csv(OUT / 'upc_rigidity_summary.csv')
store = pd.read_csv(OUT / 'store_rigidity_summary.csv')
display(upc.loc[upc.record_type.eq('distribution')])
display(store.loc[store.record_type.eq('distribution')])
display(Image(filename=str(FIG / '14_rigidity_heterogeneity.png'), width=860))

## Validation and scope boundary
The validation suite covers deterministic rerun, gap prevention, spell reconciliation, promotion-transition reconciliation, excluded-UPC protection, and source integrity. All checks reported below pass. Extreme price relatives are retained in primary results; a diagnostic comparison shows their influence on means. Econometric results are documented separately in `05_econometrics.ipynb`.

In [ ]:
validation = pd.Series(manifest['validation'], name='passed').rename_axis('test').to_frame()
display(validation)
assert validation.passed.all()
precision = pd.Series(manifest['precision_audit'], name='value').rename_axis('metric').to_frame()
display(precision)
print('Phase 4 gate:', manifest['phase4_gate'])
